# Búsqueda Exhaustiva de Combinaciones de Ensemble

**Objetivo:** evaluar **todas las combinaciones posibles** de los 5 modelos disponibles (sin iTransformer) para determinar cuál subconjunto produce el mejor ensemble.

**Modelos candidatos:** TCN, NBEATS, LSTM, TiDE, EncDec

Con 5 modelos → **2⁵ − 1 = 31 combinaciones no vacías**. Es completamente exhaustivo.

**Protocolo sin leakage (igual al notebook principal):**
- Los modelos predicen sobre el **30%** (out-of-sample genuino).
- El 30% se parte en dos:
  - `ens_fit` (50%) → optimizar pesos de cada combinación
  - `held_out` (50%) → evaluación final **nunca vista**

$$\text{df\_70 (100\%)} \xrightarrow{\text{entrenamiento}} \text{Modelos}$$
$$\text{df\_30} \xrightarrow{\text{predict}} \begin{cases} 50\% \to \text{fit ensemble weights} \\ 50\% \to \text{held-out final eval} \end{cases}$$

In [1]:
# Dependencias extra (necesarias en Colab)
!pip install -q keras-tcn pandas scikit-learn scipy dtaidistance matplotlib joblib openpyxl pyyaml


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
import importlib
import shutil
import glob
from itertools import combinations

# ---- Colab: clonar repo y localizar nueva_info/ ----
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.chdir('/content')
    if not os.path.exists('heartrate-forecasting'):
        !git clone https://github.com/AllanDBB/heartrate-forecasting.git
    os.chdir('heartrate-forecasting')

    repo_nueva = os.path.join(os.getcwd(), 'nueva_info')
    os.makedirs(repo_nueva, exist_ok=True)

    search_dirs = [
        '/content/nueva_info',
        '/content/drive/MyDrive/nueva_info',
        '/content/drive/MyDrive',
    ]
    for sdir in search_dirs:
        if not os.path.isdir(sdir):
            continue
        for f in glob.glob(os.path.join(sdir, '*.keras')):
            dst = os.path.join(repo_nueva, os.path.basename(f))
            if not os.path.exists(dst):
                shutil.copy2(f, dst)
                print(f'  Copiado: {f} -> {dst}')

    found = glob.glob(os.path.join(repo_nueva, '*.keras'))
    if len(found) < 5:
        print(f'\n⚠️  Solo se encontraron {len(found)}/5 modelos .keras')
        from google.colab import files
        uploaded = files.upload()
        for fname, data in uploaded.items():
            dest = os.path.join(repo_nueva, fname)
            with open(dest, 'wb') as fh:
                fh.write(data)
            print(f'  Guardado: {dest}')

if not IN_COLAB:
    try:
        _nb_path = __vsc_ipynb_file__
        REPO_DIR = os.path.dirname(os.path.dirname(os.path.abspath(_nb_path)))
    except NameError:
        REPO_DIR = os.path.abspath(
            os.path.join(os.path.dirname(os.path.abspath('__file__')), '..')
        )
else:
    REPO_DIR = os.getcwd()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'REPO_DIR: {REPO_DIR}')
print(f'Colab: {IN_COLAB}')

import main
import utils
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

importlib.reload(utils)
importlib.reload(main)

REPO_DIR: c:\Users\allan\OneDrive\Documentos\Asistencia 2026 I
Colab: False


<module 'main' from 'c:\\Users\\allan\\OneDrive\\Documentos\\Asistencia 2026 I\\wrappers\\..\\wrappers\\..\\main.py'>

## 1. Cargar datos y preparar ventanas supervisadas

In [3]:
INPUT_SIZE    = 200
OUTPUT_SIZE   = 200
CACHE_DIR     = 'cache_nueva_info'
ENSEMBLE_SEED = 123

main.ensure_dir(CACHE_DIR)

df_70, df_30, split_meta = main.load_split_dataframes(
    dataset_dir='dataset',
    split_seed=42,
    split_70_path='nueva_info/df_70.csv',
    split_30_path='nueva_info/df_30.csv',
)
df_70, df_30, overlap = utils.sanitize_split_dataframes(df_70, df_30)
print('Overlap eliminado:', overlap)

path_est_70 = os.path.join(CACHE_DIR, 'values_deses_70.csv')
path_est_30 = os.path.join(CACHE_DIR, 'values_deses_30.csv')
df_scaled_70, params_70 = utils.estandarizar(df_70, path_est_70)
df_scaled_30, params_30 = utils.estandarizar(df_30, path_est_30)

print(f'df_70 estandarizado: {df_scaled_70.shape}')
print(f'df_30 estandarizado: {df_scaled_30.shape}')

Usando split predefinido: nueva_info/df_70.csv | nueva_info/df_30.csv
Aviso: se excluyen columnas solapadas del split legacy para evitar leakage: ['Abel_Rodriguez_10', 'Marta_Baños_10']
Overlap eliminado: []
df_70 estandarizado: (1841, 91)
df_30 estandarizado: (1841, 39)


In [4]:
X_30, y_30, ids_30 = utils.series_to_supervised_matrix(
    df_scaled_30, input_size=INPUT_SIZE, output_size=OUTPUT_SIZE
)
print(f'Ventanas totales del 30%: X={X_30.shape}, y={y_30.shape}')

(
    X_ens_fit, X_held_out,
    y_ens_fit, y_held_out,
    ids_ens_fit, ids_held_out
) = train_test_split(
    X_30, y_30, ids_30,
    test_size=0.5,
    random_state=ENSEMBLE_SEED,
    stratify=ids_30,
)

print(f'ens_fit  : X={X_ens_fit.shape}, y={y_ens_fit.shape}')
print(f'held_out : X={X_held_out.shape}, y={y_held_out.shape}')

Ventanas totales del 30%: X=(36513, 200), y=(36513, 200)
ens_fit  : X=(18256, 200), y=(18256, 200)
held_out : X=(18257, 200), y=(18257, 200)


## 2. Cargar modelos y obtener predicciones base

> iTransformer **excluido** intencionalmente.

In [5]:
import wrappers.KerasPretrainedWrapper as _kpw_mod
import wrappers.NBeatsSupervisedWrapper as _nbeats_mod
importlib.reload(_nbeats_mod)
importlib.reload(_kpw_mod)

_nbeats_mod._get_nbeats_block_class()

from wrappers.KerasPretrainedWrapper import KerasPretrainedWrapper

# iTransformer excluido
MODEL_SPECS = {
    'TCN':    'nueva_info/tcn.keras',
    'NBEATS': 'nueva_info/nbeats.keras',
    'LSTM':   'nueva_info/lstm.keras',
    'TiDE':   'nueva_info/tide.keras',
    'EncDec': 'nueva_info/encDec.keras',
}

print('Modelos a cargar (sin iTransformer):')
for name, path in MODEL_SPECS.items():
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    print(f'  {name:10s} -> {path} ({size_mb:.1f} MB) {"OK" if exists else "MISSING"}')

Modelos a cargar (sin iTransformer):
  TCN        -> nueva_info/tcn.keras (1.2 MB) OK
  NBEATS     -> nueva_info/nbeats.keras (7.3 MB) OK
  LSTM       -> nueva_info/lstm.keras (1.1 MB) OK
  TiDE       -> nueva_info/tide.keras (86.2 MB) OK
  EncDec     -> nueva_info/encDec.keras (0.2 MB) OK


In [6]:
preds_ens_fit  = {}
preds_held_out = {}

for name, model_path in MODEL_SPECS.items():
    try:
        wrapper = KerasPretrainedWrapper(model_path, batch_size=32, name=name).load()
        preds_ens_fit[name]  = wrapper.predict(X_ens_fit)
        preds_held_out[name] = wrapper.predict(X_held_out)
        print(f'[OK] {name}: ens_fit={preds_ens_fit[name].shape}, held_out={preds_held_out[name].shape}')
    except Exception as exc:
        print(f'[SKIP] {name}: {exc}')

print(f'\nModelos disponibles: {list(preds_ens_fit.keys())}')


Modelo TCN cargado desde: nueva_info/tcn.keras
[OK] TCN: ens_fit=(18256, 200), held_out=(18257, 200)
Modelo NBEATS cargado desde: nueva_info/nbeats.keras
[OK] NBEATS: ens_fit=(18256, 200), held_out=(18257, 200)
Modelo LSTM cargado desde: nueva_info/lstm.keras
[OK] LSTM: ens_fit=(18256, 200), held_out=(18257, 200)
  TiDE explicit weight loading: 32/32 assigned
Modelo TiDE cargado desde: nueva_info/tide.keras
[OK] TiDE: ens_fit=(18256, 200), held_out=(18257, 200)
Modelo EncDec cargado desde: nueva_info/encDec.keras
[OK] EncDec: ens_fit=(18256, 200), held_out=(18257, 200)

Modelos disponibles: ['TCN', 'NBEATS', 'LSTM', 'TiDE', 'EncDec']


## 3. Desestandarizar predicciones

In [7]:
y_ens_fit_orig  = utils.desestandarizar_ventanas(y_ens_fit,  ids_ens_fit,  params_30)
y_held_out_orig = utils.desestandarizar_ventanas(y_held_out, ids_held_out, params_30)

preds_ens_fit_orig = {
    n: utils.desestandarizar_ventanas(p, ids_ens_fit,  params_30)
    for n, p in preds_ens_fit.items()
}
preds_held_out_orig = {
    n: utils.desestandarizar_ventanas(p, ids_held_out, params_30)
    for n, p in preds_held_out.items()
}

print('Desestandarización completada.')
print(f'y_held_out_orig shape: {y_held_out_orig.shape}')

Desestandarización completada.
y_held_out_orig shape: (18257, 200)


## 4. Métricas individuales (referencia)

In [8]:
individual_results = {}
for name, pred in preds_held_out_orig.items():
    individual_results[name] = utils.evaluate_all_metrics(y_held_out_orig, pred)

df_ind = pd.DataFrame(individual_results).T
df_ind.index.name = 'Modelo'
if 'MAPE' in df_ind.columns:
    df_ind = df_ind.sort_values('MAPE')
print('=== Métricas individuales (held_out) ===')
df_ind

  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 5.5178
  MAPE_median         : 4.7536
  DTW                 : 101.0097
  Pearson             : 0.7107
  Pearson_median      : 0.8602
  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 5.5617
  MAPE_median         : 4.6275
  DTW                 : 91.9022
  Pearson             : 0.719
  Pearson_median      : 0.8673
  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 6.0738
  MAPE_median         : 4.9855
  DTW                 : 104.2397
  Pearson             : 0.6588
  Pearson_median      : 0.8491
  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 5.8018
  MAPE_median         : 4.8633
  DTW                 : 98.907
  Pearson             : 0.6469
  Pearson_median      : 0.844
  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 5.5926
  MAPE_median         : 4.7035
  DTW                 : 94.2491
  Pearson             :

,MAPE,MAPE_median,DTW,Pearson,Pearson_median
Modelo,,,,,
TCN,5.5178,4.7536,101.0097,0.7107,0.8602
NBEATS,5.5617,4.6275,91.9022,0.7190,0.8673
EncDec,5.5926,4.7035,94.2491,0.6972,0.8552
TiDE,5.8018,4.8633,98.9070,0.6469,0.8440
LSTM,6.0738,4.9855,104.2397,0.6588,0.8491


## 5. Búsqueda exhaustiva de combinaciones

Probamos **todas las 31 combinaciones no vacías** de los 5 modelos.
Para cada combinación se optimizan los pesos sobre `ens_fit` y se evalúa sobre `held_out`.

Método de ensemble: **weighted average optimizado** (mismo que en el notebook principal).

In [9]:
from wrappers.EnsembleWrapper import EnsembleWrapper

available_models = list(preds_ens_fit_orig.keys())
print(f'Modelos disponibles: {available_models}')

# Generar todas las combinaciones no vacías
all_combos = []
for r in range(1, len(available_models) + 1):
    for combo in combinations(available_models, r):
        all_combos.append(combo)

print(f'Total de combinaciones a evaluar: {len(all_combos)}')
for combo in all_combos:
    print(f'  {" + ".join(combo)}')

Modelos disponibles: ['TCN', 'NBEATS', 'LSTM', 'TiDE', 'EncDec']
Total de combinaciones a evaluar: 31
  TCN
  NBEATS
  LSTM
  TiDE
  EncDec
  TCN + NBEATS
  TCN + LSTM
  TCN + TiDE
  TCN + EncDec
  NBEATS + LSTM
  NBEATS + TiDE
  NBEATS + EncDec
  LSTM + TiDE
  LSTM + EncDec
  TiDE + EncDec
  TCN + NBEATS + LSTM
  TCN + NBEATS + TiDE
  TCN + NBEATS + EncDec
  TCN + LSTM + TiDE
  TCN + LSTM + EncDec
  TCN + TiDE + EncDec
  NBEATS + LSTM + TiDE
  NBEATS + LSTM + EncDec
  NBEATS + TiDE + EncDec
  LSTM + TiDE + EncDec
  TCN + NBEATS + LSTM + TiDE
  TCN + NBEATS + LSTM + EncDec
  TCN + NBEATS + TiDE + EncDec
  TCN + LSTM + TiDE + EncDec
  NBEATS + LSTM + TiDE + EncDec
  TCN + NBEATS + LSTM + TiDE + EncDec


In [10]:
import io
import contextlib

combo_results = {}  # clave: "M1 + M2 + ..." -> dict de métricas

KEY_OPTIMIZE = 'Ensemble (Pesos Optimos)'
KEY_AVERAGE  = 'Ensemble (Promedio)'

for combo in all_combos:
    combo_name = ' + '.join(combo)
    try:
        ens = EnsembleWrapper()

        for model_name in combo:
            ens.add_model(model_name, preds_ens_fit_orig[model_name],  split='fit')
            ens.add_model(model_name, preds_held_out_orig[model_name], split='eval')

        if len(combo) == 1:
            # Modelo único: no hay qué optimizar
            pred = preds_held_out_orig[combo[0]]
            combo_results[combo_name] = utils.evaluate_all_metrics(y_held_out_orig, pred)
        else:
            # Suprimir el output verboso de compare_methods (una línea por combo)
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                ensemble_metrics = ens.compare_methods(
                    y_true_eval=y_held_out_orig,
                    y_true_fit=y_ens_fit_orig,
                    objective_metric='MAPE',
                    active_method='optimize',
                )
            # Tomar pesos optimizados; fallback a promedio simple
            if KEY_OPTIMIZE in ensemble_metrics:
                combo_results[combo_name] = ensemble_metrics[KEY_OPTIMIZE]
            else:
                combo_results[combo_name] = ensemble_metrics.get(
                    KEY_AVERAGE, next(iter(ensemble_metrics))
                )

        mape_val = combo_results[combo_name].get('MAPE', float('nan'))
        print(f'[OK] {combo_name:50s}  MAPE={mape_val:.4f}')

    except Exception as exc:
        print(f'[ERROR] {combo_name}: {exc}')

print(f'\nCombinaciones evaluadas: {len(combo_results)} / {len(all_combos)}')


  + TCN [fit]: (18256, 200)
  + TCN [eval]: (18257, 200)
  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 5.5178
  MAPE_median         : 4.7536
  DTW                 : 101.0097
  Pearson             : 0.7107
  Pearson_median      : 0.8602
[OK] TCN                                                 MAPE=5.5178
  + NBEATS [fit]: (18256, 200)
  + NBEATS [eval]: (18257, 200)
  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 5.5617
  MAPE_median         : 4.6275
  DTW                 : 91.9022
  Pearson             : 0.719
  Pearson_median      : 0.8673
[OK] NBEATS                                              MAPE=5.5617
  + LSTM [fit]: (18256, 200)
  + LSTM [eval]: (18257, 200)
  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 6.0738
  MAPE_median         : 4.9855
  DTW                 : 104.2397
  Pearson             : 0.6588
  Pearson_median      : 0.8491
[OK] LSTM                                                M

## 6. Tabla de resultados — ranking de combinaciones

In [11]:
df_combos = pd.DataFrame(combo_results).T
df_combos.index.name = 'Combinación'
df_combos['n_modelos'] = df_combos.index.map(lambda x: len(x.split(' + ')))

if 'MAPE' in df_combos.columns:
    df_combos = df_combos.sort_values('MAPE')

print(f'=== Ranking de {len(df_combos)} combinaciones (evaluado en held_out) ===')
df_combos

=== Ranking de 31 combinaciones (evaluado en held_out) ===


,MAPE,MAPE_median,DTW,Pearson,Pearson_median,n_modelos
Combinación,,,,,,
TCN + NBEATS + LSTM + TiDE + EncDec,4.9767,4.1849,84.9812,0.7741,0.8959,5
TCN + NBEATS + TiDE + EncDec,4.9859,4.1839,85.1258,0.7733,0.8938,4
NBEATS + LSTM + TiDE + EncDec,4.9884,4.1916,84.2201,0.7714,0.8953,4
NBEATS + TiDE + EncDec,5.0063,4.1884,84.1393,0.7689,0.8918,3
TCN + NBEATS + LSTM + TiDE,5.0215,4.2569,86.3941,0.7720,0.8942,4
TCN + NBEATS + TiDE,5.0435,4.2738,86.8748,0.7700,0.8918,3
NBEATS + LSTM + TiDE,5.0661,4.2735,85.4570,0.7655,0.8909,3
TCN + LSTM + TiDE + EncDec,5.0796,4.3270,87.7663,0.7604,0.8863,4
TCN + TiDE + EncDec,5.1016,4.3620,88.2096,0.7566,0.8826,3


In [12]:
# Top 10
print('=== Top 10 combinaciones por MAPE ===')
df_combos.head(10)

=== Top 10 combinaciones por MAPE ===


,MAPE,MAPE_median,DTW,Pearson,Pearson_median,n_modelos
Combinación,,,,,,
TCN + NBEATS + LSTM + TiDE + EncDec,4.9767,4.1849,84.9812,0.7741,0.8959,5
TCN + NBEATS + TiDE + EncDec,4.9859,4.1839,85.1258,0.7733,0.8938,4
NBEATS + LSTM + TiDE + EncDec,4.9884,4.1916,84.2201,0.7714,0.8953,4
NBEATS + TiDE + EncDec,5.0063,4.1884,84.1393,0.7689,0.8918,3
TCN + NBEATS + LSTM + TiDE,5.0215,4.2569,86.3941,0.7720,0.8942,4
TCN + NBEATS + TiDE,5.0435,4.2738,86.8748,0.7700,0.8918,3
NBEATS + LSTM + TiDE,5.0661,4.2735,85.4570,0.7655,0.8909,3
TCN + LSTM + TiDE + EncDec,5.0796,4.3270,87.7663,0.7604,0.8863,4
TCN + TiDE + EncDec,5.1016,4.3620,88.2096,0.7566,0.8826,3


In [13]:
# Guardar tabla completa
try:
    _root = os.path.dirname(os.path.dirname(os.path.abspath(__vsc_ipynb_file__)))
except NameError:
    _root = REPO_DIR

_cache_abs = os.path.join(_root, CACHE_DIR)
os.makedirs(_cache_abs, exist_ok=True)
csv_path = os.path.join(_cache_abs, 'combo_search_results.csv')
df_combos.to_csv(csv_path)
print(f'Guardado en: {csv_path}')

Guardado en: c:\Users\allan\OneDrive\Documentos\Asistencia 2026 I\cache_nueva_info\combo_search_results.csv


## 7. Visualizaciones

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline

if 'MAPE' not in df_combos.columns:
    print('No se encontró columna MAPE para graficar.')
else:
    fig, ax = plt.subplots(figsize=(14, max(6, len(df_combos) * 0.35)))

    # Colores según número de modelos en la combinación
    cmap = plt.cm.get_cmap('tab10', 5)
    colors = [cmap(n - 1) for n in df_combos['n_modelos']]

    bars = ax.barh(df_combos.index, df_combos['MAPE'], color=colors, edgecolor='white', height=0.7)

    # Añadir etiquetas de valor
    for bar, val in zip(bars, df_combos['MAPE']):
        ax.text(val + 0.0002, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8)

    ax.set_xlabel('MAPE (↓ mejor)', fontsize=12)
    ax.set_title('MAPE por combinación de modelos — held_out (sin leakage)', fontsize=13, fontweight='bold')
    ax.invert_yaxis()

    # Leyenda por número de modelos
    legend_patches = [
        mpatches.Patch(color=cmap(i), label=f'{i+1} modelo{"s" if i > 0 else ""}')
        for i in range(5)
    ]
    ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

    plt.tight_layout()
    plt.show()

In [ ]:
# MAPE promedio por tamaño de combinación
if 'MAPE' in df_combos.columns:
    summary = df_combos.groupby('n_modelos')['MAPE'].agg(['mean', 'min', 'max', 'count'])
    summary.columns = ['MAPE_mean', 'MAPE_min', 'MAPE_max', 'n_combinaciones']
    print('=== Resumen por tamaño de combinación ===')
    print(summary.to_string())

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(
        summary.index, summary['MAPE_mean'],
        yerr=[summary['MAPE_mean'] - summary['MAPE_min'],
              summary['MAPE_max'] - summary['MAPE_mean']],
        fmt='o-', capsize=6, linewidth=2, markersize=7, color='steelblue'
    )
    ax.set_xlabel('Número de modelos en la combinación', fontsize=11)
    ax.set_ylabel('MAPE', fontsize=11)
    ax.set_title('MAPE promedio / min / max por tamaño de combinación', fontsize=12)
    ax.set_xticks(summary.index)
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap de contribución: ¿qué modelos aparecen más en el top N?
if 'MAPE' in df_combos.columns:
    TOP_N = 10
    top_combos = df_combos.head(TOP_N)
    model_names = list(preds_ens_fit_orig.keys())

    presence = pd.DataFrame(0, index=top_combos.index, columns=model_names)
    for combo_name in top_combos.index:
        parts = combo_name.split(' + ')
        for m in parts:
            if m in model_names:
                presence.loc[combo_name, m] = 1

    fig, ax = plt.subplots(figsize=(8, max(4, TOP_N * 0.4)))
    im = ax.imshow(presence.values, aspect='auto', cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, fontsize=11)
    ax.set_yticks(range(len(top_combos)))
    ax.set_yticklabels(
        [f"{i+1}. {name}" for i, name in enumerate(top_combos.index)],
        fontsize=8
    )
    ax.set_title(f'Presencia de modelos en el Top {TOP_N} (azul = incluido)', fontsize=12)
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.04)
    plt.tight_layout()
    plt.show()

    print('\nFrecuencia de aparición en el Top', TOP_N, ':')
    print(presence.sum().sort_values(ascending=False).to_string())

In [ ]:
# Predicción de la mejor combinación vs real
if 'MAPE' in df_combos.columns:
    best_combo_name = df_combos.index[0]
    best_models = best_combo_name.split(' + ')
    print(f'Mejor combinación: {best_combo_name}')

    ens_best = EnsembleWrapper()
    for m in best_models:
        ens_best.add_model(m, preds_ens_fit_orig[m],  split='fit')
        ens_best.add_model(m, preds_held_out_orig[m], split='eval')

    if len(best_models) == 1:
        y_best_pred = preds_held_out_orig[best_models[0]]
    else:
        ens_best.compare_methods(
            y_true_eval=y_held_out_orig,
            y_true_fit=y_ens_fit_orig,
            objective_metric='MAPE',
            active_method='optimize',
        )
        y_best_pred = ens_best.predict()

    utils.plot_forecast_samples(
        y_held_out_orig, y_best_pred, n_samples=6,
        title=f'Mejor combinación: {best_combo_name} (held_out)'
    )

In [ ]:
# Error sobre el horizonte de la mejor combinación
if 'MAPE' in df_combos.columns:
    utils.plot_error_over_horizon(
        y_held_out_orig, y_best_pred,
        title=f'Error según horizonte — {best_combo_name}'
    )

## 10. Búsqueda exhaustiva con Stacking (Ridge)

Repetimos las 31 combinaciones usando **stacking** como método de ensemble.
Ridge entrena un meta-learner sobre `ens_fit` y evalúa en `held_out` — sin leakage.


In [ ]:
import io
import contextlib

KEY_STACKING = 'Ensemble (Stacking)'
stacking_results = {}

for combo in all_combos:
    combo_name = ' + '.join(combo)
    try:
        ens_s = EnsembleWrapper()

        for model_name in combo:
            ens_s.add_model(model_name, preds_ens_fit_orig[model_name],  split='fit')
            ens_s.add_model(model_name, preds_held_out_orig[model_name], split='eval')

        if len(combo) == 1:
            pred = preds_held_out_orig[combo[0]]
            stacking_results[combo_name] = utils.evaluate_all_metrics(y_held_out_orig, pred)
        else:
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                ensemble_metrics = ens_s.compare_methods(
                    y_true_eval=y_held_out_orig,
                    y_true_fit=y_ens_fit_orig,
                    objective_metric='MAPE',
                    active_method='stacking',
                )
            if KEY_STACKING in ensemble_metrics:
                stacking_results[combo_name] = ensemble_metrics[KEY_STACKING]
            else:
                stacking_results[combo_name] = ensemble_metrics.get(
                    KEY_AVERAGE, next(iter(ensemble_metrics))
                )

        mape_val = stacking_results[combo_name].get('MAPE', float('nan'))
        print(f'[OK] {combo_name:50s}  MAPE={mape_val:.4f}')

    except Exception as exc:
        print(f'[ERROR] {combo_name}: {exc}')

print(f'\nCombinaciones evaluadas: {len(stacking_results)} / {len(all_combos)}')


  + TCN [fit]: (18256, 200)
  + TCN [eval]: (18257, 200)
  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 5.5178
  MAPE_median         : 4.7536
  DTW                 : 101.0097
  Pearson             : 0.7107
  Pearson_median      : 0.8602
[OK] TCN                                                 MAPE=5.5178
  + NBEATS [fit]: (18256, 200)
  + NBEATS [eval]: (18257, 200)
  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 5.5617
  MAPE_median         : 4.6275
  DTW                 : 91.9022
  Pearson             : 0.719
  Pearson_median      : 0.8673
[OK] NBEATS                                              MAPE=5.5617
  + LSTM [fit]: (18256, 200)
  + LSTM [eval]: (18257, 200)
  Evaluation Metrics
  Ventanas evaluadas   : 18257
  MAPE                : 6.0738
  MAPE_median         : 4.9855
  DTW                 : 104.2397
  Pearson             : 0.6588
  Pearson_median      : 0.8491
[OK] LSTM                                                M

## 11. Tabla comparativa: Optimize vs Stacking


In [ ]:
df_stack = pd.DataFrame(stacking_results).T
df_stack.index.name = 'Combinación'
df_stack['n_modelos'] = df_stack.index.map(lambda x: len(x.split(' + ')))

# Unir optimize y stacking en una tabla comparativa
df_compare = pd.DataFrame({
    'MAPE_optimize': df_combos['MAPE'],
    'MAPE_stacking': df_stack['MAPE'],
    'n_modelos':     df_combos['n_modelos'],
})
df_compare['delta'] = df_compare['MAPE_optimize'] - df_compare['MAPE_stacking']
df_compare = df_compare.sort_values('MAPE_stacking')

print('=== Ranking por MAPE_stacking ===')
df_compare.head(15)


In [ ]:
# Guardar resultados de stacking
csv_stack_path = csv_path.replace('combo_search_results', 'combo_search_stacking')
df_stack.to_csv(csv_stack_path)
df_compare.to_csv(csv_path.replace('combo_search_results', 'combo_search_compare'))
print(f'Guardado: {csv_stack_path}')

# Resumen por tamaño — stacking
print('\n=== Resumen por tamaño (stacking) ===')
print(df_stack.groupby('n_modelos')['MAPE'].agg(['mean','min','max']).to_string())

# Resumen por tamaño — optimize (referencia)
print('\n=== Resumen por tamaño (optimize) ===')
print(df_combos.groupby('n_modelos')['MAPE'].agg(['mean','min','max']).to_string())
